# UniDriveVLA — Standalone Export on Google Colab

This notebook clones the repo, installs the minimal dependencies, runs
`scripts/export_standalone.py`, and verifies the ONNX output — **no mmdet /
mmcv / CUDA required**.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` (optional but faster).
> The export also works on CPU-only runtime; expect ~80 s instead of ~25 s.

---
| Output file | Size | Format |
|---|---|---|
| `exports/unidrivevla_perception.onnx` | ~1 KB (graph) | ONNX opset 14 |
| `exports/unidrivevla_perception.onnx.data` | ~113 MB | ONNX weight shard |
| `exports/unidrivevla_perception.pt` | ~126 MB | TorchScript |
| `exports/export_info.json` | ~1 KB | Metadata |


## Step 1 — Clone the repository

In [ ]:
import os

REPO = "https://github.com/ARNiteshKumar/UniDriveVLA_MulticoreWare.git"
BRANCH = "claude/nuscenes-mini-dataset-repo-V4wLx"
REPO_DIR = "/content/UniDriveVLA_MulticoreWare"

if not os.path.isdir(REPO_DIR):
    !git clone --depth=1 --branch {BRANCH} {REPO} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

%cd {REPO_DIR}
!git log --oneline -5

## Step 2 — Install dependencies

Only `torch`, `torchvision`, `onnx`, and `onnxruntime` are needed.
Colab already ships with `torch` and `torchvision`, so only `onnx` and
`onnxruntime` need installing (takes < 30 s).

In [ ]:
import importlib, sys

def _need(pkg):
    return importlib.util.find_spec(pkg) is None

to_install = []
if _need("onnx"):        to_install.append("onnx")
if _need("onnxruntime"): to_install.append("onnxruntime")
if _need("onnxscript"):  to_install.append("onnxscript")   # needed by torch >= 2.1

if to_install:
    pkgs = " ".join(to_install)
    !pip install --quiet {pkgs}
    print(f"Installed: {pkgs}")
else:
    print("All dependencies already present.")

import torch, torchvision, onnx, onnxruntime
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"onnx         {onnx.__version__}")
print(f"onnxruntime  {onnxruntime.__version__}")
print(f"Device       {'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Step 3 — Run the export

`export_standalone.py` performs four steps internally:
1. Builds ResNet-50 + FPN + UnifiedPerceptionDecoder (32.2 M params, random weights)
2. Runs a reference forward pass and prints output shapes
3. Exports ONNX (opset 14) and verifies against PyTorch reference outputs
4. Exports TorchScript via `torch.jit.trace`

In [ ]:
import os
os.makedirs("exports", exist_ok=True)

!python scripts/export_standalone.py \
    --output-dir exports/ \
    --img-h 450 \
    --img-w 800 \
    --num-cams 6

## Step 4 — Verify exported files

In [ ]:
import os, json
from pathlib import Path

export_dir = Path("exports")
print("Exported files:")
for f in sorted(export_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<45} {size_mb:>8.2f} MB")

print()
info_path = export_dir / "export_info.json"
if info_path.exists():
    info = json.loads(info_path.read_text())
    print("export_info.json:")
    print(f"  Model      : {info['model']}")
    print(f"  ONNX opset : {info['onnx_opset']}")
    print(f"  Outputs    :")
    for name, meta in info["outputs"].items():
        print(f"    {name:<14}: shape={meta['shape']}")

## Step 5 — Run inference with the ONNX model

Verifies that `onnxruntime` can load and execute the exported model.

In [ ]:
import time
import numpy as np
import onnxruntime as ort

ONNX_PATH = "exports/unidrivevla_perception.onnx"

providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
sess = ort.InferenceSession(ONNX_PATH, providers=providers)
print(f"Active providers : {sess.get_providers()}")

# Dummy input: (batch=1, cameras=6, RGB, H=450, W=800)
dummy = np.zeros((1, 6, 3, 450, 800), dtype=np.float32)

t0 = time.time()
outputs = sess.run(None, {"img": dummy})
elapsed = (time.time() - t0) * 1000

OUTPUT_NAMES = ["det_cls", "det_bbox", "map_cls", "map_pts", "plan_trajs", "plan_scores"]
print(f"\nInference time: {elapsed:.0f} ms\n")
print(f"  {'Output':<16} {'Shape':<25} {'Min':>8}  {'Max':>8}")
print("  " + "-" * 60)
for name, out in zip(OUTPUT_NAMES, outputs):
    print(f"  {name:<16} {str(list(out.shape)):<25} {out.min():>8.4f}  {out.max():>8.4f}")

## Step 6 — Run inference with the TorchScript model

In [ ]:
import time
import torch

TS_PATH = "exports/unidrivevla_perception.pt"
device  = "cuda" if torch.cuda.is_available() else "cpu"

model = torch.jit.load(TS_PATH, map_location=device)
model.eval()
print(f"TorchScript model loaded on: {device}")

dummy_t = torch.zeros(1, 6, 3, 450, 800, device=device)

# Warm-up
with torch.no_grad():
    _ = model(dummy_t)

t0 = time.time()
with torch.no_grad():
    ts_outputs = model(dummy_t)
elapsed = (time.time() - t0) * 1000

OUTPUT_NAMES = ["det_cls", "det_bbox", "map_cls", "map_pts", "plan_trajs", "plan_scores"]
print(f"\nInference time ({device}): {elapsed:.0f} ms\n")
print(f"  {'Output':<16} {'Shape':<25}")
print("  " + "-" * 42)
for name, out in zip(OUTPUT_NAMES, ts_outputs):
    print(f"  {name:<16} {str(list(out.shape)):<25}")

## Step 7 — Use `verify_export.py` (verbose output)

In [ ]:
# Full verbose verification: top-5 detections, planning trajectory, map counts
!python scripts/verify_export.py --model exports/unidrivevla_perception.onnx

## Step 8 — Download exported files (optional)

Download any of the generated files to your local machine.

In [ ]:
try:
    from google.colab import files

    # Uncomment the file(s) you want to download:
    files.download("exports/export_info.json")
    # files.download("exports/unidrivevla_perception.onnx")       # graph only (tiny)
    # files.download("exports/unidrivevla_perception.onnx.data")  # weights ~113 MB
    # files.download("exports/unidrivevla_perception.pt")         # TorchScript ~126 MB
    print("Download triggered.")
except ImportError:
    print("Not running in Colab — files are at:", [str(p) for p in Path("exports").iterdir()])

---
## Notes

| Item | Detail |
|---|---|
| Weights | Random initialisation (no checkpoint) — outputs are structurally valid but semantically meaningless |
| ONNX verify tolerance | max_diff < 1e-4 (typically < 3e-7) between PyTorch and ONNX Runtime |
| Real checkpoint | Download from `xiaomi-research/UniDriveVLA` on HuggingFace, then pass `--checkpoint path/to/ckpt.pth` |
| Full evaluation | Requires mmdet3d stack — see `docs/installation.md` and `scripts/setup_env.sh` |
| Export script | `scripts/export_standalone.py` — no mmdet/mmcv needed |
